In [23]:
import nibabel as nib
import os
import matplotlib.pyplot as plt
import numpy as np
from dipy.io.gradients import read_bvals_bvecs
from fury import window, actor

# Data and Class imports
# from data import data, affine, labels, bvals, bvecs, vox_size
from env_main_updated import UnifiedTractographyEnv
from agent import BranchingStreamlineAgent,UnifiedBranchingAgent
from dipy.viz import colormap
from dipy.tracking import utils

In [32]:
data_path="sub-1001"
for folder in os.listdir(data_path):
    print(folder)

anat
dti
dwi
fodf
mask
sh
subdivisions
tractography


In [28]:
anat_img=nib.load(r"sub-1001\anat\sub-1001__T1w.nii.gz")
affine=anat_img.affine
data=anat_img.get_fdata()

In [36]:
vox_size=[1.0,1.0,1.0]

In [33]:
dwi_path=os.path.join(data_path,'dwi')
dwi_bval_path=r"sub-1001\dwi\sub-1001__dwi.bval"
dwi_bvec_path=r"sub-1001\dwi\sub-1001__dwi.bvec"
dwi_data_path=r"sub-1001\dwi\sub-1001__dwi.nii.gz"
bvals,bvecs=read_bvals_bvecs(fbvals=dwi_bval_path,fbvecs=dwi_bvec_path)
dwi_img=nib.load(dwi_data_path)
dwi_data=dwi_img.get_fdata()

In [7]:
mask_path=os.path.join(data_path,"mask")
coords=[]
for i,m in enumerate(os.listdir(mask_path)):
    mask_data_path=os.path.join(mask_path,m)
    mask_data=nib.load(mask_data_path).get_fdata()
    mask=mask_data.astype(bool)
    coords.append(  np.array(np.where(mask)).T)
    print("mask name:",m,"shape: ",coords[i].shape)


mask name: sub-1001__mask_csf.nii.gz shape:  (299572, 3)
mask name: sub-1001__mask_gm.nii.gz shape:  (572500, 3)
mask name: sub-1001__mask_wm.nii.gz shape:  (382567, 3)


In [35]:
scene=window.Scene()

p=actor.point(points=coords[5],colors=(1,0,0))
scene.add(p)
window.show(scene)

In [25]:
seed_mask=np.zeros(mask_data.shape,dtype=bool)

In [26]:
seed_mask.shape

(135, 166, 100)

In [33]:
seed_mask[50:80, : ,:]=mask[50:80, : ,:]

In [34]:
coords.append(  np.array(np.where(seed_mask)).T)

In [67]:
coords[5].shape

(120741, 3)

In [8]:
import nibabel as nib
import numpy as np

# 1. Load your masks (assuming they are binary 0 or 1)
wm_mask = nib.load(os.path.join(mask_path,'sub-1001__mask_wm.nii.gz')).get_fdata()
gm_mask = nib.load(os.path.join(mask_path,'sub-1001__mask_gm.nii.gz')).get_fdata()
csf_mask = nib.load(os.path.join(mask_path,'sub-1001__mask_csf.nii.gz')).get_fdata()

# 2. Initialize the labels array with zeros (Background)
labels = np.zeros(wm_mask.shape, dtype=np.int32)

# 3. Assign Tissue Types
labels[csf_mask > 0.5] = 1  # CSF
labels[gm_mask > 0.5] = 3   # Gray Matter
labels[wm_mask > 0.5] = 2   # White Matter (Initial assignment)

# 4. Refine Seeds (Subset of White Matter)
# If you want to seed only from the center of the brain (e.g., Corpus Callosum)
# we can mask out everything except a central box in the WM
z_mid = wm_mask.shape[2] // 2
seed_mask = np.zeros_like(wm_mask)
# Create a slab 10 voxels thick in the middle
seed_mask[:, :, z_mid-5 : z_mid+5] = 1

# Update labels: Only WM voxels in that central slab become '2' (Seeds)
# The rest of the WM stays as a different label (e.g., 4) or remains 2
final_labels = labels.copy()
final_labels[(labels == 2) & (seed_mask == 0)] = 4 # Non-seed White Matter
final_labels[(labels == 2) & (seed_mask == 1)] = 2 # Actual Seeds

# 5. Save the new label map
new_label_img = nib.Nifti1Image(final_labels, nib.load(os.path.join(mask_path,'sub-1001__mask_wm.nii.gz')).affine)
nib.save(new_label_img, 'processed_labels.nii.gz')

In [24]:
preprocessed_labels=nib.load('processed_labels.nii.gz')
labels_data=preprocessed_labels.get_fdata()
labels_data.shape

(135, 166, 100)

In [25]:
coords=np.argwhere(labels_data==2)
coords.shape

(67655, 3)

In [26]:
n_samples=10000
indices = np.random.choice(coords.shape[0], n_samples, replace=False)
reduced_matrix = coords[indices]
reduced_matrix.shape


(10000, 3)

In [29]:
data.shape

(135, 166, 100)

In [34]:
bvecs.shape

(65, 3)

In [37]:
env = UnifiedTractographyEnv(
    data=dwi_data, affine=affine, labels=labels_data, 
    bvals=bvals, bvecs=bvecs, vox_size=vox_size,
    fa_threshold=0.25, max_curvature_deg=45
)

In [38]:
agent = UnifiedBranchingAgent(env)

In [ ]:
seed_indices = utils.seeds_from_mask((labels_data == 2),affine=np.eye(4),density=(1,1,1))
print(seed_indices.shape)

(67655, 3)


In [39]:
seed_indices=reduced_matrix
#seed_indices = utils.seeds_from_mask((labels_data == 2),affine=np.eye(4),density=(1,1,1))
np.random.shuffle(seed_indices)
num_seeds =  len(seed_indices)
scene = window.Scene()
all_streamlines = []

print(f"Starting branching tracking for {num_seeds} seeds...")

for i in range(num_seeds):
    seed = seed_indices[i].astype(np.float32)
    
    # This returns a LIST of streamlines because of branching
    branches = agent.track(seed)
    
    if branches:
        all_streamlines.extend(branches)
        print(f"Seed {i+1}: Found {len(branches)} branches.")

# 4. Rendering in Fury
if all_streamlines:
    print(f"Total streamlines generated: {len(all_streamlines)}")
    
    # Use coloring based on local orientation for visual clarity
    env.render_streamlines(scene,all_streamlines)
    env.render_wm_surface(scene)
    
    # Add the white matter surface at low opacity for anatomical context
    # env.render_wm_surface(scene, opacity=0.05)
 
    scene.reset_camera()
    
    print("Opening interactive window...")
    window.show(scene, size=(1024, 768), title="Branching Fiber Tracking")
else:
    print("No streamlines found. Check peak thresholds.")

Starting branching tracking for 10000 seeds...
Seed 1: Found 2 branches.
Seed 2: Found 1 branches.
Seed 3: Found 1 branches.
Seed 4: Found 1 branches.
Seed 5: Found 1 branches.
Seed 6: Found 1 branches.
Seed 7: Found 1 branches.
Seed 8: Found 1 branches.
Seed 9: Found 1 branches.
Seed 10: Found 2 branches.
Seed 11: Found 4 branches.
Seed 12: Found 1 branches.
Seed 13: Found 2 branches.
Seed 14: Found 53 branches.
Seed 15: Found 1 branches.
Seed 16: Found 1 branches.
Seed 17: Found 733 branches.
Seed 18: Found 1 branches.
Seed 19: Found 13 branches.
Seed 20: Found 1 branches.
Seed 21: Found 1 branches.
Seed 22: Found 2 branches.
Seed 23: Found 4 branches.
Seed 24: Found 27 branches.
Seed 25: Found 1 branches.
Seed 26: Found 1 branches.
Seed 27: Found 7 branches.
Seed 28: Found 1 branches.
Seed 30: Found 1 branches.
Seed 31: Found 1 branches.
Seed 32: Found 9 branches.
Seed 33: Found 1 branches.
Seed 34: Found 2 branches.
Seed 35: Found 1 branches.
Seed 36: Found 1 branches.
Seed 38: Fou

In [50]:
scene=window.Scene()
env.render_bval_bvec(scene,labels_data==3)
window.show(scene)

In [40]:
env.render_seeds_mask(scene)
window.show(scene)

In [41]:
from dipy.io.streamline import save_trk
from data import hardi_img
from dipy.io.stateful_tractogram import Space, StatefulTractogram
sft = StatefulTractogram(all_streamlines, hardi_img, Space.RASMM)
save_trk(sft, "trk/sub1001_10000.trk", all_streamlines)

In [1]:
from nibabel.streamlines import load as load_trk
trk=load_trk( "trk/sub1001_10000.trk")
streamlines=trk.streamlines

In [5]:
scene = window.Scene()
color = colormap.line_colors(streamlines)
scene.add(actor.line(streamlines, color))
# Save a snapshot
window.record(scene=scene, out_path='sub1001_10000.png', size=(900, 900))
# Interactive display (works if display available)
window.show(scene)

C:\Users\Samsung\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py:3508: UserWarning: We'll no longer accept the way you call the line function in future versions of FURY.

Here's how to call the Function line: line(lines_value, colors='value', opacity='value', linewidth='value', spline_subdiv='value', lod='value', lod_points='value', lod_points_size='value', lookup_colormap='value', depth_cue='value', fake_tube='value')

  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
import os
print(os.listdir("../"))

['.vscode', '3D_rendering', 'basics', 'datasets', 'dMRI_reconstruction-main.pdf', 'fiberSnake', 'main', 'output_tracks.trk', 'pyTorch', 'Screen Recording 2025-10-25 003550.mp4', 'shear', 'shear.zip', 'trk', 'trk.zip']


In [6]:
from nibabel.streamlines import load as load_trk
trk=load_trk( r"../output_tracks.trk")
streamlines=trk.streamlines

In [3]:
from fury import window, actor

In [4]:
from dipy.viz import colormap

In [7]:
scene = window.Scene()
color = colormap.line_colors(streamlines)
scene.add(actor.line(streamlines, color))
# Save a snapshot
# window.record(scene=scene, out_path='output_tracks.png', size=(900, 900))
# Interactive display (works if display available)
window.show(scene)

In [13]:
scene=window.Scene()
env.render_seeds_mask(scene)
env.render_wm_surface(scene)
window.show(scene,size=(800,800))

In [1]:
import nibabel as nib
from dipy.viz import window, actor, colormap as cmap

print("Load your generated track file") 
trk = nib.streamlines.load("output_tracks.trk")
streamlines = trk.streamlines

print("Create the 3D scene")
scene = window.Scene()

print("Add the tracks (colored by local orientation)") 
streamline_actor = actor.line(streamlines, cmap.line_colors(streamlines))
scene.add(streamline_actor)

print(" Show the interactive window")
window.show(scene, size=(800, 800))

KeyboardInterrupt: 

In [3]:
import numpy as np
print(np.__version__)

1.26.4


In [10]:
import nibabel as nib

In [5]:
from dipy.viz import window, actor, colormap as cmap

KeyboardInterrupt: 

In [6]:
import dipy

In [7]:
from dipy import viz

KeyboardInterrupt: 

In [8]:
import fury

In [9]:
from fury import window, actor